# 02 — Inspect a calibration result

Five static plots (Agg backend, headless-safe) that together tell you whether
a calibration converged well and where any residual distortion lives.

Install with `pip install 'midas-auto-calibrate[viz]'` to get matplotlib.

In [1]:
import shutil, tempfile
from pathlib import Path

import midas_auto_calibrate as mac
from midas_auto_calibrate.viz import static as viz

## Run a calibration first

Same as notebook 01 — skip this cell if you already have a `CalibrationResult`.

In [2]:
workdir = Path(tempfile.mkdtemp(prefix='mac_viz_'))
img = workdir / 'CeO2_00001.tif'
shutil.copy(mac.data.CEO2_PILATUS, img)
shutil.copy(mac.data.CEO2_PILATUS_DARK, workdir / 'dark.tif')
shutil.copy(mac.data.CEO2_PILATUS_MASK, workdir / 'mask_upd.tif')

result = mac.auto_calibrate(
    mac.CalibrationConfig(
        material='CeO2',
        lattice_params=(5.4116,) * 3 + (90.0,) * 3,
        wavelength=0.172973, pixel_size=172.0,
        lsd=657_436.9, ybc=685.5, zbc=921.0,
        nr_pixels_y=1475, nr_pixels_z=1679,
        dark_file='dark.tif', mask_file='mask_upd.tif',
        im_trans_opt=[2],
        n_iterations=5,
    ),
    img, work_dir=workdir, n_cpus=4,
)
print(f'pseudo_strain = {result.pseudo_strain:.2f} µε')

pseudo_strain = 2151.37 µε


## Convergence — per-iteration MeanStrain

A clean monotonic drop means the optimizer is healthy. Plateaus or jumps
typically point at ring-outlier or doublet issues.

In [3]:
viz.convergence(result);

## Rings overlay — ideal rings on the calibrant image

Dashed circles are the refined-geometry ring positions; cyan dots are the
per-point fits from ``corr.csv``. Misaligned rings = bad calibration.

In [4]:
viz.rings_overlay(result, img);

## Residual heatmap — ΔR over (ring, η)

ΔR is fitted minus ideal in µm. A well-calibrated detector shows noise-level
residuals; structured patterns (diagonal bands, k=2 sinusoid) indicate
unmodelled distortion.

In [5]:
viz.residual_heatmap(result);

## Fourier harmonics — per-ring FFT of ΔR(η)

Bright columns flag specific residual distortion modes:
- **k=2**: residual tilt (ty or tz undercorrected)
- **k=4**: astigmatism / p2 residual
- **k=6**: trefoil
- **k=8**: octupole

In [6]:
viz.fourier_harmonics(result);

## Distortion field — |ΔR| at each fitted point

Spatial map of where residuals live on the detector. Hot spots near panel
edges typically indicate panel-shift or per-panel distortion issues.

In [7]:
viz.distortion_field(result);

## All five plots as one PNG bundle

`viz.inspect()` writes all five as separate PNGs — handy for papers, PRs,
or beamline logbook entries. Skips plots whose inputs aren't available.

In [8]:
written = viz.inspect(result, image=img, out_dir=workdir / 'plots', prefix='ceo2')
for name, path in written.items():
    print(f'  {name:>11} -> {path}')

  convergence -> /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/mac_viz_wou8kqtj/plots/ceo2_convergence.png
        rings -> /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/mac_viz_wou8kqtj/plots/ceo2_rings.png
     residual -> /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/mac_viz_wou8kqtj/plots/ceo2_residual.png
      fourier -> /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/mac_viz_wou8kqtj/plots/ceo2_fourier.png
   distortion -> /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/mac_viz_wou8kqtj/plots/ceo2_distortion.png
